# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2 Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install mlcroissant matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id`.


In [ ]:
# List available record sets from the Croissant dataset
record_sets = metadata.record_sets

if not record_sets:
    print("No record sets found in the schema metadata. Attempting to autodiscover available record sets by inspecting dataset.")

    # Sometimes the metadata may not include record_sets, so we try to list resources
    available = dataset._get_record_sets()
    print("Discovered record set @ids:")
    for rs in available:
        print(f"- {rs['@id']} (name: {rs.get('name', 'N/A')})")
else:
    print("Record sets defined in metadata:")
    for rs in record_sets:
        print(f"- {rs['@id']} (name: {rs.get('name', 'N/A')})")

# For this dataset as of publication, the main tabular record set is:
# '@id': 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'
main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'

# Print fields and columns in this record set
main_rs = dataset._get_record_set(main_record_set_id)
fields = main_rs.get('fields', main_rs.get('field', []))

print(f"\nFields for record set '{main_record_set_id}':")
for f in fields:
    print(f"- {f['@id']}: {f.get('name', f.get('rdfs:label', ''))} (type: {f.get('dataType', '')})")
    if 'column' in f:
        if isinstance(f['column'], list):
            for col in f['column']:
                print(f"   - column @id: {col['@id']}")
        else:
            print(f"   - column @id: {f['column']['@id']}")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. All field IDs are referenced by their `@id`.


In [ ]:
# Extract data from the primary record set using @id
record_sets = [main_record_set_id]
dataframes = {}

for record_set in record_sets:
    print(f"Loading records for {record_set} ...")
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)

df = dataframes[main_record_set_id]
print("Available columns (@id):")
print(list(df.columns))
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing: filter by abundance, normalize a metric, group by a field. Some field `@id`s, based on extracted schema, are used below (ensure validity by printing columns above).

In [ ]:
# Select a numeric field for analysis.
# Example: Use 'abundance_EV1' (commonly, try columns with 'abundance', 'coverage', or 'peptide_count').

# Replace below with exact @id of abundance/numeric column from printed columns above.
possible_fields = [col for col in df.columns if 'abundance' in col.lower() or 'coverage' in col.lower() or 'count' in col.lower()]
print("Possible numeric fields for analysis:", possible_fields)

# Example selection
numeric_field = None
for cand in ['abundance_EV1', 'abundance_EV2', 'peptide_count']:
    if cand in df.columns:
        numeric_field = cand
        break

# Fallback: pick first numeric column
if numeric_field is None and possible_fields:
    numeric_field = possible_fields[0]
print(f"Using numeric_field = {numeric_field}")

if numeric_field is None:
    print("No numeric field found for analysis.")
else:
    # Ensure type is numeric
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].quantile(0.75)
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (Q3): {len(filtered_df)} records")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by accession or any categorical column
    group_field = None
    for cand in ['accession', 'modification', 'description']:
        if cand in filtered_df.columns:
            group_field = cand
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
    else:
        print("No group field (accession/modification/description) found for grouping.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and its relationship to other fields (e.g., histogram, boxplot, scatterplot).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot
    plt.figure(figsize=(6,2))
    sns.boxplot(x=df[numeric_field])
    plt.title(f"Boxplot of {numeric_field}")
    plt.show()

    # If group_field exists and is not too high cardinality
    if 'group_field' in locals() and group_field and filtered_df[group_field].nunique() < 15:
        plt.figure(figsize=(10,4))
        sns.barplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field} (Top 15)")
        plt.xticks(rotation=45)
        plt.show()

else:
    print("No numeric field available to plot.")

## 6. Conclusion
We explored the FAIR^2 mass spectrometry dataset using its Croissant schema. We loaded the main protein table, listed all available field `@id`s, performed filtering and normalization on a selected abundance metric, grouped by key identifiers, and visualized the distribution of numeric data. For further analysis, consult the croissant schema for field descriptions and perform deeper domain-specific analyses on the protein modifications and abundance data.